# C16_E02 — Detector de stiction sobre datos del lazo

**Capítulo:** 16 — Machine Learning, Edge AI y supervisión inteligente de lazos  
**Identificador:** `C16_E02`  
**Objetivo pedagógico:** Diagnóstico inteligente complementario al clásico (Capítulo 10).  
**Librerías:** scikit-learn, numpy

## Ejemplo industrial

Distinguir lazos con stiction de lazos sanos en una unidad con muchos lazos PID.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Generación de señales de lazo y extracción de features

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
np.random.seed(0)

def features(y):
    return [
        np.var(y),
        np.mean(np.abs(np.diff(y))),
        float(np.max(y) - np.min(y)),
        float(np.percentile(np.abs(np.fft.rfft(y - np.mean(y))), 95)),
    ]

# Genera N curvas de lazo: la mitad sano, la otra mitad con stiction (cuadrada)
N_curvas = 200; N_pts = 200
X = np.zeros((N_curvas, 4)); y = np.zeros(N_curvas, dtype=int)
t = np.linspace(0, 20, N_pts)
for i in range(N_curvas):
    if i < N_curvas//2:
        # Lazo sano: oscilación senoidal pequeña + ruido
        s = 0.05*np.sin(2*np.pi*0.1*t) + 0.05*np.random.randn(N_pts)
        y[i] = 0
    else:
        # Stiction: cuadrada de mayor amplitud
        s = 0.3*np.sign(np.sin(2*np.pi*0.15*t)) + 0.05*np.random.randn(N_pts)
        y[i] = 1
    X[i] = features(s)

## 2. Clasificador binario

In [2]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
print(f"Accuracy: {accuracy_score(y_te, y_pred):.3f}")
print(classification_report(y_te, y_pred, target_names=['sano','stiction']))

Accuracy: 1.000
              precision    recall  f1-score   support

        sano       1.00      1.00      1.00        29
    stiction       1.00      1.00      1.00        31

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



## 3. Interpretación

Un clasificador supervisado entrenado con etiquetas (sano / stiction) reconoce el patrón temporal del lazo a partir de cuatro features simples (varianza, derivada media absoluta, rango, energía espectral). Con datos sintéticos suficientes y separables, la accuracy es alta. **Riesgo industrial:** el clasificador es tan bueno como sus etiquetas; en planta, la fiabilidad depende de mantener un dataset etiquetado actualizado. Decisión humana antes de actuar (Capítulo 17).